<div style="font-size: 0.85em;">

<p><strong>Function calling</strong> allows the model to return structured arguments for a tool/function you define, instead of free text.</p>

<p>You define a Pydantic model that describes the arguments a function expects. The model is then given that schema and can return valid JSON arguments to call that function.</p>

<p>In LangChain, we use <code>bind_tools()</code> to attach one or more tool schemas to a chat model. The model can then choose to call a tool and provide arguments.</p>

<p><strong>Why it matters for RAG:</strong></p>
<ul>
  <li>Trigger retrieval when the user asks a query.</li>
  <li>Return structured search parameters (e.g., query, top_k, language).</li>
  <li>Provide structured responses with citations.</li>
</ul>

<pre>
Define Tool Schema (Pydantic model)
        |
        v
bind_tools([schema]) to chat model
        |
        v
User question
        |
        v
Model returns tool call arguments (JSON)
        |
        v
You execute the tool/function with those arguments
</pre>

</div>

In [1]:
from langchain_openai import ChatOpenAI
from pydantic import BaseModel, Field
from dotenv import load_dotenv

In [2]:
# Load environment variables
load_dotenv()

# Define a tool schema for a search function
class SearchQuery(BaseModel):
    query: str = Field(description='The search query string')
    top_k: int = Field(description='Number of results to return', default=5)
    
print('Schema created successfully')

Schema created successfully


Bind Tools to the Model

In [4]:
# Create the chat model
llm = ChatOpenAI(model='gpt-4o-mini', temperature=0)

# Bind the tool schema to the model
llm_with_tools = llm.bind_tools([SearchQuery])

print('Tools bound to model successfully.')

Tools bound to model successfully.


Invoke and Inspect Tool Call

In [14]:
# Ask the model a question that should trigger a tool call
response = llm_with_tools.invoke('Search for RAG tutorials, top 3')

print('Response')
print(response)
print()

# Check if the model decided to call a tool
if response.tool_calls:
    tool_call = response.tool_calls[0]
    print(f'\nTool name: {tool_call["name"]}')
    print(f'Arguments: {tool_call["args"]}')
    
else:
    print('No tool call made.')

Response
content='' additional_kwargs={'tool_calls': [{'id': 'call_QEuSh4UYpA6T0NJjVQXGExKL', 'function': {'arguments': '{"query":"RAG tutorials","top_k":3}', 'name': 'SearchQuery'}, 'type': 'function'}], 'refusal': None} response_metadata={'token_usage': {'completion_tokens': 21, 'prompt_tokens': 69, 'total_tokens': 90, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_name': 'gpt-4o-mini-2024-07-18', 'system_fingerprint': 'fp_b1b99e3b92', 'finish_reason': 'tool_calls', 'logprobs': None} id='run-638da7f5-e3c9-49a2-8546-5d0c55ed0be3-0' tool_calls=[{'name': 'SearchQuery', 'args': {'query': 'RAG tutorials', 'top_k': 3}, 'id': 'call_QEuSh4UYpA6T0NJjVQXGExKL', 'type': 'tool_call'}] usage_metadata={'input_tokens': 69, 'output_tokens': 21, 'total_tokens': 90}


Tool name: SearchQuery
Arguments: {'query': 'RAG tutorials', 'top_k': 3

Simulate Executing the Tool

In [19]:
# Define a simple search function

def fake_search(query: str, top_k: int = 5):
    
    dummy_results = [
        'Introduction to RAG',
        'Advanced RAG Techniques',
        'Building RAG with LangChain',
        'RAG Evaluation Methods',
        'Multilingual RAG Systems'
    ]
    
    return dummy_results[:top_k]


# Extract arguments from the tool call
if response.tool_calls:
    args = response.tool_calls[0]['args']
    #print(args)
    print(f'  query = {args["query"]}')
    print(f'  top_k = {args["top_k"]}')
    
    print()
    results = fake_search(**args)
    print('\nSearch results:')
    
    print(results)
    
    for i, res in enumerate(results, start=1):
        print(f'{i}. {res}')

else:
    print('No tool call to execute.')

  query = RAG tutorials
  top_k = 3


Search results:
['Introduction to RAG', 'Advanced RAG Techniques', 'Building RAG with LangChain']
1. Introduction to RAG
2. Advanced RAG Techniques
3. Building RAG with LangChain
